### Tools

Models can requests to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema including the name of the tool, a description of what it does, and/or argument definations (often a JSON schema).
2. A function that implements the tool's functionality.

In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(model="openai/gpt-oss-20b")
response = model.invoke("write a poem about the beauty of nature in 100 words")
for chunk in response.content:
    print(chunk,end="", flush=True)


Morning dew clings to emerald blades,  
Sunrise paints the sky in gold.  
River whispers secrets to stone,  
Leaves dance in breezes bold.  

Mountains stand in quiet grace,  
Their peaks kiss clouds above.  
Wildflowers bloom in colors bright,  
Nature's palette, pure love.  

Nightfall drapes a velvet veil,  
Stars twinkle in endless blue.  
Every breath, a hymn of life,  
In wonder, hearts renew.  

Fields roll like silver seas,  
Birds compose symphonies on wing.  
Crickets sing beneath moonlit trees,  
Echoing songs that time will bring.  

In every stone, a story hidden,  
Life's rhythm beats beneath skin,  
We wander, learn, bind  
With earth's hymn.

In [4]:
### to create a tool
### we import tools from langchain

from langchain.tools import tool

# we use this tool as a decorator to create a tool that can be used by the model to call a function that we define.
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"The weather at {location} is sunny and 20 degrees."


#bind the tool to the model
model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Tool args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call the get_weather function with location "New York".', 'tool_calls': [{'id': 'fc_b0a0a350-0c24-4527-b770-61888f20adab', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 128, 'total_tokens': 167, 'completion_time': 0.042794077, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007285456, 'prompt_tokens_details': None, 'queue_time': 0.286285021, 'total_time': 0.050079533}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1de3a-d945-745d-b2e8-2abfeb8456f7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_b0a0a350-0c24-4527-b770-61888f20adab', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'

### Tool Execution loop

When a model calls a tool, the tool is executed and the result is returned to the model. The model can then use the result to generate a response to the user.

In [6]:
#step 1: Model generats tool calls
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print(f"AI message is: {ai_msg}")
print(f"Message is: {messages}")
#step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    
#step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weather in New York is sunny and 20 degrees.

AI message is: content='' additional_kwargs={'reasoning_content': 'We need to call the get_weather function with location "New York".', 'tool_calls': [{'id': 'fc_c72b4397-455e-4e15-8da0-3dcf4d5e7c27', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 128, 'total_tokens': 167, 'completion_time': 0.042256756, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00614869, 'prompt_tokens_details': None, 'queue_time': 0.320588347, 'total_time': 0.048405446}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1de3a-db2a-70a4-81a2-e15965237f59-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_c72b4397-455e-4e15-8da0-3dcf4d5e7c27', 'type': 'tool_call'}] invalid_tool_calls=[] usa

In [7]:
print(messages)

[{'role': 'user', 'content': 'What is the weather in New York?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the get_weather function with location "New York".', 'tool_calls': [{'id': 'fc_c72b4397-455e-4e15-8da0-3dcf4d5e7c27', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 128, 'total_tokens': 167, 'completion_time': 0.042256756, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00614869, 'prompt_tokens_details': None, 'queue_time': 0.320588347, 'total_time': 0.048405446}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1de3a-db2a-70a4-81a2-e15965237f59-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_c72b4397-455e-4e15-8d